In [0]:
import requests
import json
from datetime import datetime, timezone

# Call the API
resp = requests.get("https://api.carbonintensity.org.uk/intensity", timeout=10)
resp.raise_for_status()  # stops the notebook loudly if the API errors, instead of silently saving bad data

# Wrap the raw response with metadata about WHEN we pulled it
# This timestamp is what lets you tell data apart if you run this multiple times a day
record = {
    "ingested_at": datetime.now(timezone.utc).isoformat(),
    "raw_json": json.dumps(resp.json())
}

# Turn it into a Spark DataFrame - just one row, containing the whole raw response as text
df = spark.createDataFrame([record])

# Append (not overwrite) - Bronze should accumulate history over time, never lose old pulls
df.write.mode("append").saveAsTable("carbon_intensity.bronze.intensity_raw")

print(f"Ingested 1 record at {record['ingested_at']}")